<a href="https://colab.research.google.com/github/mahamtaqi3-cloud/Predicting-Behavioral-Phenotypes-from-Genomic-Variants-in-Drosophila-using-Machine-Learning/blob/main/Predicting_Behavioral_Phenotypes_from_Genomic_Variants_in_Drosophila_using_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-allel

In [2]:
import os
if os.path.exists('/content/drive'):
    print("Drive is mounted!")
else:
    print("Drive is not mounted. Run the mount command.")

Drive is mounted!


In [3]:
!wget -O dgrp2.vcf.gz https://zenodo.org/records/155396/files/dgrp2.vcf.gz?download=1

--2026-06-27 15:00:45--  https://zenodo.org/records/155396/files/dgrp2.vcf.gz?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 193799443 (185M) [application/octet-stream]
Saving to: ‘dgrp2.vcf.gz’

dgrp2.vcf.gz        100%[===================>] 184.82M   751KB/s    in 4m 39s  

2026-06-27 15:05:26 (678 KB/s) - ‘dgrp2.vcf.gz’ saved [193799443/193799443]



In [4]:
!gunzip dgrp2.vcf.gz

gzip: dgrp2.vcf already exists; do you wish to overwrite (y or n)? ^C


In [5]:
!head -n 20 dgrp2.vcf

##fileformat=VCFv4.1
##source=DGRP2
##reference=dm3
##INFO=<ID=REFCOUNT,Number=1,Type=Integer,Description="Reference Allele Count">
##INFO=<ID=ALTCOUNT,Number=1,Type=Integer,Description="Alternative Allele Count">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	line_21	line_26	line_28	line_31	line_32	line_38	line_40	line_41	line_42	line_45	line_48	line_49	line_57	line_59	line_69	line_73	line_75	line_83	line_85	line_88	line_91	line_93	line_100	line_101	line_105	line_109	line_129	line_136	line_138	line_142	line_149	line_153	line_158	line_161	line_176	line_177	line_181	line_189	line_195	line_208	line_217	line_223	line_227	line_228	line_229	line_233	line_235	line_237	line_239	line_256	line_280	line_287	line_301	line_303	line_304	line_306	line_307	line_309	line_310	line_313	line_315	line_317	line_318	line_319	line_320	line_321	line_324	line_325	line_332	line_335	line_336	line_338	line_340	line_348	line_350	line_352	line_354	line_355

In [7]:
import allel
# Load the genotype calls (GT) from the VCF
callset = allel.read_vcf('dgrp2.vcf')
g = allel.GenotypeArray(callset['calldata/GT'])

print(f"Genotype matrix shape: {g.shape}")

Genotype matrix shape: (4438427, 205, 2)


In [8]:
# Convert to a simple 0, 1, 2 matrix
# This tells the computer: 0 = Ref/Ref, 1 = Ref/Alt, 2 = Alt/Alt
X = g.to_n_alt(fill=0)
print(f"Feature matrix (X) shape: {X.shape}")

Feature matrix (X) shape: (4438427, 205)


In [9]:
# Check the sample names in the VCF
vcf_samples = callset['samples']
print(vcf_samples)

['line_21' 'line_26' 'line_28' 'line_31' 'line_32' 'line_38' 'line_40'
 'line_41' 'line_42' 'line_45' 'line_48' 'line_49' 'line_57' 'line_59'
 'line_69' 'line_73' 'line_75' 'line_83' 'line_85' 'line_88' 'line_91'
 'line_93' 'line_100' 'line_101' 'line_105' 'line_109' 'line_129'
 'line_136' 'line_138' 'line_142' 'line_149' 'line_153' 'line_158'
 'line_161' 'line_176' 'line_177' 'line_181' 'line_189' 'line_195'
 'line_208' 'line_217' 'line_223' 'line_227' 'line_228' 'line_229'
 'line_233' 'line_235' 'line_237' 'line_239' 'line_256' 'line_280'
 'line_287' 'line_301' 'line_303' 'line_304' 'line_306' 'line_307'
 'line_309' 'line_310' 'line_313' 'line_315' 'line_317' 'line_318'
 'line_319' 'line_320' 'line_321' 'line_324' 'line_325' 'line_332'
 'line_335' 'line_336' 'line_338' 'line_340' 'line_348' 'line_350'
 'line_352' 'line_354' 'line_355' 'line_356' 'line_357' 'line_358'
 'line_359' 'line_360' 'line_361' 'line_362' 'line_365' 'line_367'
 'line_370' 'line_371' 'line_373' 'line_374' 'line_

In [10]:
# Create a mapping from 'line_XX' (VCF format) to 'DGRP_XX' (Phenotype format)
sample_map = {name: f"DGRP_{name.split('_')[1]}" for name in vcf_samples}

# Rename the samples in your VCF callset
# This aligns your genomic data with your phenotype names
aligned_samples = [sample_map[name] for name in vcf_samples]
callset['samples'] = aligned_samples

In [11]:
import os
print(os.listdir('/content/'))

['.config', 'dgrp2.vcf.gz', 'dgrp2.vcf', 'drive', 'dgrp2_filtered.vcf', 'sample_data']


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!find /content -name "S29_Aggression_GSEI_summary_measures.tsv"

In [14]:
!find /content/drive -name "S29_Aggression_GSEI_summary_measures.tsv"

In [15]:
import os

# Search your entire mounted Drive for your specific files
target_files = ['S29_Aggression_GSEI_summary_measures.tsv',
                'S15_cv_WakingActivity_summary_measures.tsv',
                'S13_Food_Intake_summary_measures.tsv']

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file in target_files:
            print(f"FOUND: {os.path.join(root, file)}")

In [16]:
import os

# This looks through your entire Drive for the files
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if "Aggression" in file: # Looking for one of your files
            print(f"THE FULL PATH IS: {os.path.join(root, file)}")

THE FULL PATH IS: /content/drive/MyDrive/S29_Aggression_GSEI_summary_mean.tsv


In [17]:
import pandas as pd
import os

# Define the folder path
base_path = '/content/drive/MyDrive'

# Use the EXACT filenames confirmed by your system search
path_agg = os.path.join(base_path, 'S29_Aggression_GSEI_summary_mean.tsv')
# Note: Ensure you check the exact names for your other files as well!
# If the others have '_measures' in them, keep that,
# but if they also have 'mean', change them to match.

# Load the data
df_aggression = pd.read_csv(path_agg, sep='\t')
print("Successfully loaded!")
print(df_aggression.head())

Successfully loaded!
       DGRP sex     value
0  DGRP_032   M -0.376040
1  DGRP_038   M  0.271323
2  DGRP_042   M -0.180635
3  DGRP_045   M  0.845892
4  DGRP_059   M  0.254606


In [18]:
import os

# Search Drive for the other two files
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if "Waking" in file or "Food" in file:
            print(f"FOUND: {file}")

FOUND: S15_cv_WakingActivity_summary_mean.tsv
FOUND: S13_Food_Intake_summary_mean.tsv


In [19]:
import pandas as pd
import os

# 1. Define your base path
base_path = '/content/drive/MyDrive'

# 2. Use the EXACT filenames confirmed by your search
path_waking = os.path.join(base_path, 'S15_cv_WakingActivity_summary_mean.tsv')
path_food = os.path.join(base_path, 'S13_Food_Intake_summary_mean.tsv')

# 3. Load the remaining data
df_waking = pd.read_csv(path_waking, sep='\t')
df_food = pd.read_csv(path_food, sep='\t')

# 4. Merge all three dataframes on 'DGRP'
# We merge aggression with waking, then merge the result with food
master_df = df_aggression.merge(df_waking, on='DGRP').merge(df_food, on='DGRP')

# 5. Display the result
print("All phenotypic data loaded and merged successfully!")
print(master_df.head())

All phenotypic data loaded and merged successfully!
       DGRP sex_x   value_x sex_y    value_y sex     value
0  DGRP_038     M  0.271323     F  16.426629   F  1.169618
1  DGRP_038     M  0.271323     F  16.426629   M  1.147569
2  DGRP_038     M  0.271323     M  21.008048   F  1.169618
3  DGRP_038     M  0.271323     M  21.008048   M  1.147569
4  DGRP_042     M -0.180635     F  21.887462   F  2.003750


In [20]:
# Save the merged dataframe to your Drive as a new .tsv file
master_df.to_csv('/content/drive/MyDrive/master_phenotype_data.tsv', sep='\t', index=False)
print("master_df saved to Google Drive!")

master_df saved to Google Drive!


In [21]:
!pip install scikit-allel

In [22]:
import allel

# Load your VCF file
vcf_path = '/content/dgrp2.vcf' # Update this if it's in a different folder
callset = allel.read_vcf(vcf_path)

# Inspect the samples and variants
print("Samples in VCF:", callset['samples'])
print("Number of variants:", len(callset['variants/POS']))

Samples in VCF: ['line_21' 'line_26' 'line_28' 'line_31' 'line_32' 'line_38' 'line_40'
 'line_41' 'line_42' 'line_45' 'line_48' 'line_49' 'line_57' 'line_59'
 'line_69' 'line_73' 'line_75' 'line_83' 'line_85' 'line_88' 'line_91'
 'line_93' 'line_100' 'line_101' 'line_105' 'line_109' 'line_129'
 'line_136' 'line_138' 'line_142' 'line_149' 'line_153' 'line_158'
 'line_161' 'line_176' 'line_177' 'line_181' 'line_189' 'line_195'
 'line_208' 'line_217' 'line_223' 'line_227' 'line_228' 'line_229'
 'line_233' 'line_235' 'line_237' 'line_239' 'line_256' 'line_280'
 'line_287' 'line_301' 'line_303' 'line_304' 'line_306' 'line_307'
 'line_309' 'line_310' 'line_313' 'line_315' 'line_317' 'line_318'
 'line_319' 'line_320' 'line_321' 'line_324' 'line_325' 'line_332'
 'line_335' 'line_336' 'line_338' 'line_340' 'line_348' 'line_350'
 'line_352' 'line_354' 'line_355' 'line_356' 'line_357' 'line_358'
 'line_359' 'line_360' 'line_361' 'line_362' 'line_365' 'line_367'
 'line_370' 'line_371' 'line_373' '

In [23]:
# 1. Normalize DGRP column to match VCF format (e.g., DGRP_038 -> line_38)
# We remove 'DGRP_' and leading zeros, then add 'line_' prefix
master_df['line_id'] = master_df['DGRP'].apply(lambda x: f"line_{int(x.split('_')[1])}")

# 2. Check for matches
vcf_samples = list(callset['samples'])
common_samples = master_df[master_df['line_id'].isin(vcf_samples)]

print(f"Total phenotypic samples: {len(master_df)}")
print(f"Total VCF samples: {len(vcf_samples)}")
print(f"Samples matching in both: {len(common_samples)}")

# 3. Sort both to ensure order is identical
common_samples = common_samples.sort_values('line_id').reset_index(drop=True)

Total phenotypic samples: 280
Total VCF samples: 205
Samples matching in both: 280


In [24]:
# Filter for biallelic SNPs, remove rare variants (MAF > 0.05)
!bcftools view -m2 -M2 -v snps -q 0.05 /content/dgrp2.vcf > /content/dgrp2_filtered.vcf

[W::vcf_parse] Contig '2L' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '2R' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '3L' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '3R' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '4' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig 'X' is not defined in the header. (Quick workaround: index the file with tabix.)


In [25]:
!apt-get install bcftools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bcftools is already the newest version (1.13-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [26]:
# Filter the VCF:
# -m2 -M2 ensures we keep only biallelic SNPs
# -v snps ensures we only keep single nucleotide polymorphisms
# -q 0.05 removes rare variants (keeps those with MAF > 5%)
!bcftools view -m2 -M2 -v snps -q 0.05 /content/dgrp2.vcf > /content/dgrp2_filtered.vcf

[W::vcf_parse] Contig '2L' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '2R' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '3L' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '3R' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '4' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig 'X' is not defined in the header. (Quick workaround: index the file with tabix.)


In [2]:
# Instead of using all, pick a random subset to speed up testing
# Only do this if you have hundreds of thousands of variants
subset_size = 20000
indices_variants = np.random.choice(genotypes.shape[0], subset_size, replace=False)
X_subset = genotypes[indices_variants, :].T
X_imputed = imputer.fit_transform(X_subset)

In [3]:
from sklearn.decomposition import PCA

# 1. Start with a smaller subset of variants to avoid immediate OOM
subset_size = 10000
indices_variants = np.random.choice(genotypes.shape[0], subset_size, replace=False)
X_subset = genotypes[indices_variants, :].T

# 2. Impute only this subset
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_subset)

# 3. Reduce dimensions using PCA
# This keeps the most "informative" variance while shrinking the feature space
pca = PCA(n_components=100)
X_reduced = pca.fit_transform(X_imputed)

# 4. Train on the reduced, manageable feature set
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print(f"R-squared score: {r2_score(y_test, model.predict(X_test)):.4f}")

R-squared score: 0.4547


In [4]:
# Add this after model.fit(X_train, y_train)
importances = model.feature_importances_
# Note: Since you used PCA, you'll be looking at the importance of the components

In [1]:
import os
import gc
import pandas as pd
import allel
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Paths and Data Configuration
base_path = '/content/drive/MyDrive'
vcf_path = '/content/dgrp2_filtered.vcf'
pheno_files = {
    'Aggression': 'S29_Aggression_GSEI_summary_mean.tsv',
    'Waking_Activity': 'S15_cv_WakingActivity_summary_mean.tsv',
    'Food_Intake': 'S13_Food_Intake_summary_mean.tsv'
}

# 1. Load VCF headers once (minimal RAM)
vcf_samples = list(allel.read_vcf_headers(vcf_path).samples)

for name, file in pheno_files.items():
    print(f"\n--- Starting: {name} ---")

    # 2. Align phenotype to VCF samples
    df_pheno = pd.read_csv(os.path.join(base_path, file), sep='\t')
    df_pheno['line_id'] = df_pheno['DGRP'].apply(lambda x: f"line_{int(x.split('_')[1])}")
    common_samples = df_pheno[df_pheno['line_id'].isin(vcf_samples)].sort_values('line_id')
    sample_indices = [vcf_samples.index(s) for s in common_samples['line_id']]

    # 3. Read and subset efficiently
    callset = allel.read_vcf(vcf_path, fields=['calldata/GT'])
    gt = allel.GenotypeArray(callset['calldata/GT'])

    # Immediately subset to needed samples and convert
    X = gt.take(sample_indices, axis=1).to_n_alt(fill=0).T

    # IMPORTANT: Delete massive source objects immediately
    del callset, gt, sample_indices
    gc.collect() # Force garbage collection

    # 4. Dimension Reduction (PCA)
    # Using a subset of features (first 10,000) if still crashing
    pca = PCA(n_components=10)
    X_reduced = pca.fit_transform(X[:, :10000]) # Limit to top 10k variants for RAM
    del X
    gc.collect()

    # 5. Train and validate
    y = common_samples['value'].values
    X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    print(f"{name} R-squared: {r2_score(y_test, model.predict(X_test)):.4f}")

    # Final wipe for this iteration
    del model, X_train, X_test, y_train, y_test, X_reduced, y
    gc.collect()


--- Starting: Aggression ---
Aggression R-squared: -0.0324

--- Starting: Waking_Activity ---
Waking_Activity R-squared: -0.3351

--- Starting: Food_Intake ---
Food_Intake R-squared: 0.1286


In [5]:
for name, file in pheno_files.items():
    print(f"\n--- Starting: {name} ---")

    # 1. Load phenotype
    df_pheno = pd.read_csv(os.path.join(base_path, file), sep='\t')
    df_pheno['line_id'] = df_pheno['DGRP'].apply(lambda x: f"line_{int(x.split('_')[1])}")

    # 2. Get samples and indices
    vcf_samples = list(allel.read_vcf_headers(vcf_path).samples)
    common_samples = df_pheno[df_pheno['line_id'].isin(vcf_samples)].sort_values('line_id')
    sample_indices = [vcf_samples.index(s) for s in common_samples['line_id']]

    # 3. Read and subset efficiently
    callset = allel.read_vcf(vcf_path, fields=['calldata/GT'])
    gt = allel.GenotypeArray(callset['calldata/GT'])
    X = gt.take(sample_indices, axis=1).to_n_alt(fill=0).T
    del callset, gt, sample_indices
    gc.collect()

    # 4. Dimension Reduction
    pca = PCA(n_components=10)
    X_reduced = pca.fit_transform(X[:, :10000])
    del X
    gc.collect()

    # 5. Train and validate
    y = common_samples['value'].values
    X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # 6. Feature Importance Extraction (The code you just asked for)
    importances = model.feature_importances_
    # Calculate impact
    # Note: Using pca.components_ directly works here
    final_importance = pd.Series(np.abs(pca.components_[:10].T @ importances), index=[f"var_{i}" for i in range(10000)])

    print(f"{name} R-squared: {r2_score(y_test, model.predict(X_test)):.4f}")
    print(f"Top 5 variants for {name}:\n{final_importance.sort_values(ascending=False).head(5)}")

    del model, X_train, X_test, y_train, y_test, X_reduced, y
    gc.collect()


--- Starting: Aggression ---
Aggression R-squared: -0.0693
Top 5 variants for Aggression:
var_807    0.013570
var_808    0.012689
var_805    0.012638
var_806    0.012638
var_803    0.012638
dtype: float64

--- Starting: Waking_Activity ---
Waking_Activity R-squared: -0.5306
Top 5 variants for Waking_Activity:
var_7081    0.013786
var_7079    0.013640
var_7210    0.012983
var_7080    0.012861
var_7095    0.012832
dtype: float64

--- Starting: Food_Intake ---
Food_Intake R-squared: 0.1536
Top 5 variants for Food_Intake:
var_3256    0.011898
var_3255    0.011898
var_4087    0.011373
var_3257    0.011349
var_4072    0.010951
dtype: float64


In [6]:
# Extract variant positions from the VCF
variants_pos = allel.read_vcf(vcf_path, fields=['variants/CHROM', 'variants/POS'])
# Now, 'var_3256' corresponds to:
chrom = variants_pos['variants/CHROM'][3256]
pos = variants_pos['variants/POS'][3256]

print(f"var_3256 is located at {chrom}:{pos}")

var_3256 is located at 2L:526758


In [16]:
# Re-run your main training loop
for name, file in pheno_files.items():
    # ... (your existing model training code) ...

    # After you define 'final_importance', add these 3 lines:
    top_n = final_importance.sort_values(ascending=False).head(5)
    top_n.to_csv(f"{name}_top_variants.csv")
    print(f"Exported {name}_top_variants.csv")

Exported Aggression_top_variants.csv
Exported Waking_Activity_top_variants.csv
Exported Food_Intake_top_variants.csv


In [17]:
# List the indices of your top variants for a specific phenotype
# For example, your top 5 for Food Intake:
top_indices = [3256, 3255, 4087, 3257, 4072]

print("Copy this list into FlyBase:")
for idx in top_indices:
    # Fetch the exact chromosome and position from your VCF data
    chrom = callset['variants/CHROM'][idx]
    pos = callset['variants/POS'][idx]
    print(f"{chrom}:{pos}")

Copy this list into FlyBase:
2L:526758
2L:526756
2L:575813
2L:526805
2L:575334
